# Swimtrends — exploring the curated zone

Interactive analysis over the Spec 2 curated Parquet, read live from S3 via DuckDB.

**Kernel:** pick **`Swimtrends (st-scrape)`** (top-right kernel picker) — that is the project `.venv` with `duckdb`, `pandas`, `matplotlib`.

**Credentials:** `loader.connect()` defaults `AWS_PROFILE=swimtrends`, so S3 reads resolve `~/.aws/credentials` automatically. The first run downloads DuckDB's `httpfs`/`aws` extensions.

**Vocabulary:** stroke is Danish — `Fri`/`Ryg`/`Bryst`/`Fly`/`IM`/`HM`; `course` is `LCM`/`SCM`; `gender` is `M`/`F`. Times stored as both `*_time` (formatted string) and `*_centiseconds` (numeric — use this for math/plots).

In [ ]:
# Locate the st-scrape dir (the one holding the `analytics` package) regardless
# of where the kernel started, put it on sys.path, and make it the working dir.
import sys, os
from pathlib import Path

def _find_st_scrape(start):
    for base in [start, *start.parents]:
        for cand in (base, base / 'st-scrape'):
            if (cand / 'analytics' / 'loader.py').exists():
                return cand
    raise RuntimeError('Could not find st-scrape (the dir with analytics/loader.py)')

ST_SCRAPE = _find_st_scrape(Path.cwd())
if str(ST_SCRAPE) not in sys.path:
    sys.path.insert(0, str(ST_SCRAPE))
os.chdir(ST_SCRAPE)
print('st-scrape:', ST_SCRAPE)

In [ ]:
# Open a DuckDB connection bound to the curated zone with every view loaded.
from analytics import loader
import pandas as pd

pd.set_option('display.max_rows', 100)
con = loader.connect()

# Run SQL and get a DataFrame back (renders as a rich table; ready for plotting).
def q(sql):
    return con.sql(sql).df()

q('SELECT count(*) AS curated_rows FROM cur_obt')

## 1. What's in the curated zone

The dataset is still being backfilled, so start by seeing what's actually there before filtering. If a later query comes back empty, widen or change the `category` / `season` / event filters to match what these cells show.

In [ ]:
# Coverage by season.
q('''
SELECT season,
       count(DISTINCT meet_id) AS meets,
       count(*)                AS individual_swims
FROM individual_results
GROUP BY season
ORDER BY season
''')

In [ ]:
# Which championship categories exist (the trend key), and how much data each has.
q('''
SELECT category,
       count(DISTINCT meet_id) AS meets,
       count(*)                AS swims
FROM results_by_category
GROUP BY category
ORDER BY swims DESC
''')

## 2. Best times

`personal_best` = fastest individual swim per swimmer / event / course, across all seasons.

In [ ]:
q('''
SELECT name, best_time, points, meet_name, season
FROM personal_best
WHERE stroke = 'Bryst' AND distance = 200 AND course = 'LCM'
ORDER BY best_centiseconds
LIMIT 15
''')

## 3. Headline trend — how fast to make the final

`final_cutline_by_season` = the 8th-fastest preliminary swim (the 8-lane final cut-line) per championship / event / season. Lower seconds = faster, so a falling line means the standard is getting tougher.

Adjust `category` to one that showed up in section 1 if this comes back empty.

In [ ]:
cut = q('''
SELECT season, gender,
       cutline_centiseconds / 100.0 AS cutline_seconds,
       cutline_time
FROM final_cutline_by_season
WHERE category = 'DM-L' AND distance = 200 AND stroke = 'Bryst' AND course = 'LCM'
ORDER BY season, gender
''')
cut

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
for gender, grp in cut.groupby('gender'):
    grp = grp.sort_values('season')
    ax.plot(grp['season'], grp['cutline_seconds'], marker='o', label=gender)
ax.set_title('DM-L 200m Bryst (LCM) — time to make the 8-lane final')
ax.set_xlabel('Season')
ax.set_ylabel('Cut-line (seconds, lower = faster)')
ax.legend(title='Gender')
ax.grid(True, alpha=0.3)
plt.show()

## 4. How the whole field moves

`event_standard_by_season` gives the full distribution per season — best, median, quartiles, and the top-8 average — so you can see whether an event is improving across the board or just at the top.

In [ ]:
std = q('''
SELECT season, gender,
       best_cs     / 100.0 AS best_s,
       top8_avg_cs / 100.0 AS top8_avg_s,
       median_cs   / 100.0 AS median_s,
       swims
FROM event_standard_by_season
WHERE category = 'DM-L' AND distance = 200 AND stroke = 'Bryst' AND course = 'LCM'
ORDER BY season, gender
''')
std

In [ ]:
men = std[std['gender'] == 'M'].sort_values('season')
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(men['season'], men['best_s'],     marker='o', label='Best')
ax.plot(men['season'], men['top8_avg_s'], marker='s', label='Top-8 avg')
ax.plot(men['season'], men['median_s'],   marker='^', label='Median')
ax.set_title('DM-L 200m Bryst (LCM), men — event standard by season')
ax.set_xlabel('Season')
ax.set_ylabel('Seconds (lower = faster)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 5. Your turn

Any view from the catalog (see `docs/analytics.md`) or raw SQL over `cur_obt`. `con.sql(...)` prints a result; `q(...)` returns a DataFrame you can plot or `.to_csv(...)`.

In [ ]:
q('''
SELECT name, club, points, completed_time, meet_name
FROM event_leaderboard
WHERE distance = 100 AND stroke = 'Fri' AND course = 'LCM'
  AND season = (SELECT max(season) FROM event_leaderboard)
  AND points_rank <= 10
ORDER BY gender, points_rank
''')